# Amazon Review Intelligence
### Un asistente local para consultar y comprender reseñas de compradores

#### **William Suaza**
#### **Anderson Trujillo**
#### **Bairon Gutierrez**

##### Universidad ICESI
##### Tarea 6

**Proyecto aplicado de procesamiento de lenguaje natural** · 17 de octubre de 2026

Este trabajo presenta un asistente en español que permite formular preguntas sobre reseñas de productos Amazon en lenguaje natural. El usuario puede buscar testimonios relevantes, solicitar cifras concretas e interpretar comentarios sin recorrer manualmente cientos de registros.

El desarrollo se ejecuta en **Windows 11**, con modelos locales servidos por **Ollama**, recuperación semántica mediante **FAISS**, un servidor **MCP** con dos herramientas y una interfaz **Gradio**. Dos Skills locales orientan la interpretación y revisan la evidencia de las respuestas.

## 1. De las reseñas a las preguntas

Una reseña contiene opiniones, dificultades y detalles útiles que suelen quedar dispersos entre muchos comentarios. En lugar de construir otro clasificador de sentimiento, decidí trabajar con un asistente que responda preguntas específicas y muestre los textos que respaldan sus respuestas.

El corpus se obtuvo de **SetFit/amazon_reviews_multi_es**. Seleccionamos una muestra fija de **1.000 reseñas**, con **200 por cada calificación de una a cinco estrellas**, para disponer de casos variados. Esta decisión permite revisar consultas de distintas calificaciones, pero significa que los porcentajes se refieren únicamente a la muestra; **no representan la distribución real de las reseñas de Amazon**.

El trabajo se centra en la recuperación y consulta de documentos. 

## 2. Cómo está organizada la solución

El recorrido de una pregunta es el siguiente:

`Gradio → agente Python + Qwen3 → MCP → búsqueda FAISS o estadísticas pandas → validación → respuesta con evidencia`

El servidor MCP ofrece **exactamente dos herramientas**:

| Herramienta | Función dentro del asistente |
|---|---|
| `buscar_resenas` | Recupera reseñas relevantes por similitud semántica y puede restringirlas por estrellas. Entrega ID, calificación y texto original. |
| `estadisticas_resenas` | Calcula conteos y porcentajes exactos sobre la muestra, con filtros por calificación y coincidencia textual literal. |

Los embeddings se producen con **EmbeddingGemma (768 dimensiones)** y se consultan en un índice FAISS. **Qwen3 4B** interpreta preguntas y redacta explicaciones breves. Las cifras y las citas originales se incorporan directamente desde MCP, no se dejan al criterio del modelo.

Las **Skills** son procedimientos en archivos `SKILL.md` que el programa Python descubre y aplica cuando corresponde. No son herramientas MCP adicionales ni se cargan automáticamente en Ollama.

La cantidad de reseñas y en general todos los componentes el proyecto se acotaron para poder ser ejecutados desde un equipo local. 

## 3. Preparación del entorno

Los componentes se ejecutan localmente en `C:\Users\bairo\Amazon_MCP_RAG`. Antes de abrir este notebook se instala el **entorno virtual de Python** y se descargan los dos modelos de Ollama. Los scripts de preparación e indexación están incluidos en el proyecto.

Para abrir Jupyter desde PowerShell:

```powershell
cd $HOME\Amazon_MCP_RAG
.\.venv\Scripts\python.exe -m jupyter lab
```

Si se abre el notebook desde otra carpeta, debe ajustarse `RAIZ` en la siguiente celda.

In [1]:
from pathlib import Path
import sys, subprocess, socket, time, json
import pandas as pd
import faiss

RAIZ = Path.cwd().resolve()
REQUERIDOS = [
    '01_preparar_resenas.py', '02_construir_indice.py',
    '04_servidor_mcp_v3_2.py', '07_app_gradio_v4_skills.py',
    '08_gestor_skills.py', '09_probar_skills.py',
    'skills/analisis-voz-consumidor/SKILL.md',
    'skills/auditoria-evidencia-rag/SKILL.md',
]
faltantes = [f for f in REQUERIDOS if not (RAIZ / f).is_file()]
print('Carpeta:', RAIZ)
print('Python:', sys.version.split()[0])
print('Archivos esenciales faltantes:', faltantes or 'ninguno')
assert not faltantes, 'Completa los archivos del proyecto antes de continuar.'

Carpeta: C:\Users\bairo\Amazon_MCP_RAG
Python: 3.12.4
Archivos esenciales faltantes: ninguno


### Modelos disponibles

Se emplean **Qwen3 4B Instruct** para interpretar y redactar, y **EmbeddingGemma** para representar las reseñas como vectores. La aplicación de escritorio de Ollama debe estar ejecutándose.

In [2]:
import ollama

MODELO_LLM = 'qwen3:4b-instruct-2507-q4_K_M'
MODELO_EMBED = 'embeddinggemma:300m-qat-q4_0'
modelos = [str(m.model) for m in ollama.list().models]
print('Modelos locales:', *modelos, sep='\n - ')
assert MODELO_LLM in modelos, 'Falta el modelo de conversación en Ollama.'
assert MODELO_EMBED in modelos, 'Falta el modelo de embeddings en Ollama.'

Modelos locales:
 - embeddinggemma:300m-qat-q4_0
 - qwen3:4b-instruct-2507-q4_K_M
 - llama3.2:3b


## 4. Corpus e índice de búsqueda

Para que la ejecución sea repetible, el corpus se guarda en `data/resenas.csv` y el índice vectorial en `vectorstore/resenas.faiss`, acompañado de `config.json`. 

Se seleccionó FAISS por ser una solución ligera que permite realizar búsquedas semánticas directamente desde Python, sin necesidad de desplegar y mantener un servidor de base de datos vectorial independiente. Esta decisión resulta especialmente adecuada para las características de nuestro equipo y el volumen de información utilizado, permitiendo simplificar la arquitectura y reducir la sobrecarga operativa.

La siguiente celda **reutiliza** los archivos si ya existen. Solo prepara el corpus o genera los embeddings cuando faltan los respectivos archivos. Si se cambia el CSV, también deberá regenerarse el índice, pues ambos comparten el orden de las filas.

In [3]:
CORPUS = RAIZ / 'data' / 'resenas.csv'
INDICE = RAIZ / 'vectorstore' / 'resenas.faiss'
CONFIG = RAIZ / 'vectorstore' / 'config.json'

if not CORPUS.exists():
    print('Preparando corpus (requiere conexión inicial o caché del dataset)...')
    subprocess.run([sys.executable, str(RAIZ / '01_preparar_resenas.py')], cwd=RAIZ, check=True)
else:
    print('Corpus existente: se reutiliza.')

if not INDICE.exists() or not CONFIG.exists():
    print('Generando índice de embeddings...')
    subprocess.run([sys.executable, str(RAIZ / '02_construir_indice.py')], cwd=RAIZ, check=True)
else:
    print('Índice FAISS existente: se reutiliza.')

Corpus existente: se reutiliza.
Índice FAISS existente: se reutiliza.


In [4]:
corpus = pd.read_csv(CORPUS, dtype={'id': 'string'}, encoding='utf-8-sig')
config = json.loads(CONFIG.read_text(encoding='utf-8'))
indice = faiss.read_index(str(INDICE))
assert len(corpus) == 1000 and corpus['id'].is_unique
assert set(['id','text','estrellas']).issubset(corpus.columns)
assert indice.ntotal == len(corpus) and indice.d == int(config['dimension']) == 768
print('Reseñas:', len(corpus))
print('IDs únicos:', corpus['id'].nunique())
print('Dimensiones vectoriales:', indice.d)
print('Reseñas por calificación:', corpus['estrellas'].value_counts().sort_index().to_dict())

Reseñas: 1000
IDs únicos: 1000
Dimensiones vectoriales: 768
Reseñas por calificación: {1: 200, 2: 200, 3: 200, 4: 200, 5: 200}


## 5. Dos herramientas publicadas mediante MCP

MCP permite que el asistente descubra las herramientas disponibles mediante un protocolo común, en lugar de llamar sus funciones internas directamente. En este caso el transporte elegido es **HTTP local**, en `127.0.0.1:8000/mcp`.

Si el servidor ya está escuchando, la celda de inicio no crea una segunda instancia. La verificación posterior confirma que se trata de la versión que admite filtros por estrellas.

In [5]:
import subprocess, socket, time

def puerto_activo(host, puerto, espera=0.5):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(espera)
        return s.connect_ex((host, puerto)) == 0

PROCESO_MCP = globals().get('PROCESO_MCP', None)
if puerto_activo('127.0.0.1', 8000):
    print('Servidor en el puerto 8000: se verificará el catálogo antes de utilizarlo.')
else:
    with open(RAIZ / 'mcp_servidor.log', 'a', encoding='utf-8') as log:
        PROCESO_MCP = subprocess.Popen(
            [sys.executable, str(RAIZ / '04_servidor_mcp_v3_2.py')],
            cwd=str(RAIZ), stdout=log, stderr=subprocess.STDOUT,
            creationflags=getattr(subprocess, 'CREATE_NEW_PROCESS_GROUP', 0)
        )
    for _ in range(40):
        if puerto_activo('127.0.0.1', 8000):
            break
        if PROCESO_MCP.poll() is not None:
            raise RuntimeError('Servidor MCP no pudo iniciar; revisa mcp_servidor.log.')
        time.sleep(0.5)
    assert puerto_activo('127.0.0.1', 8000), 'MCP no respondió en el puerto 8000.'
    print('MCP iniciado. PID:', PROCESO_MCP.pid)

Servidor en el puerto 8000: se verificará el catálogo antes de utilizarlo.


In [6]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def verificar_mcp():
    async with streamablehttp_client('http://127.0.0.1:8000/mcp') as (r, w, _):
        async with ClientSession(r, w) as sesion:
            await sesion.initialize()
            herramientas = (await sesion.list_tools()).tools
            nombres = {h.name for h in herramientas}
            assert nombres == {'buscar_resenas', 'estadisticas_resenas'}
            buscador = next(h for h in herramientas if h.name == 'buscar_resenas')
            assert 'estrellas' in buscador.inputSchema.get('properties', {}), \
                'Está activo un servidor MCP antiguo; requiere V3.2.'
            for h in herramientas:
                print(h.name, '→', list(h.inputSchema.get('properties', {})))
            return nombres

catalogo = await verificar_mcp()
print('MCP verificado:', sorted(catalogo))

buscar_resenas → ['pregunta', 'top_k', 'estrellas']
estadisticas_resenas → ['estrellas', 'palabra_clave']
MCP verificado: ['buscar_resenas', 'estadisticas_resenas']


### Una comprobación directamente contra las herramientas

Además del catálogo, esta consulta pequeña confirma que MCP ejecuta ambas operaciones sobre los datos reales: una búsqueda semántica con filtro por estrellas y un conteo literal.

In [7]:
async def comprobar_consultas_mcp():
    async with streamablehttp_client('http://127.0.0.1:8000/mcp') as (r, w, _):
        async with ClientSession(r, w) as sesion:
            await sesion.initialize()
            busqueda = await sesion.call_tool('buscar_resenas', arguments={
                'pregunta': 'retrasos en la entrega', 'top_k': 1, 'estrellas': [1]
            })
            cifras = await sesion.call_tool('estadisticas_resenas', arguments={
                'estrellas': [1], 'palabra_clave': 'entrega'
            })
            assert not busqueda.isError and not cifras.isError
            def datos(respuesta):
                return respuesta.structuredContent or json.loads(respuesta.content[0].text)
            fuentes = datos(busqueda)
            cantidad = datos(cifras)
            if list(fuentes) == ['result']:
                fuentes = fuentes['result']
            if list(cantidad) == ['result']:
                cantidad = cantidad['result']
            print('ID recuperado:', fuentes['fuentes'][0]['id'])
            print('Calificación:', fuentes['fuentes'][0]['estrellas'])
            print('Coincidencias literales:', cantidad['reseñas_filtradas'])
            print('Porcentaje en la muestra:', cantidad['porcentaje_sobre_la_muestra'], '%')

await comprobar_consultas_mcp()

ID recuperado: es_0262483
Calificación: 1
Coincidencias literales: 3
Porcentaje en la muestra: 0.3 %


## 6. Incorporación de las dos Skills

La primera Skill, **Análisis de voz del consumidor**, indica cómo interpretar opiniones sin añadir detalles ausentes. La segunda, **Auditoría de evidencia RAG**, define los controles de trazabilidad: los IDs deben existir, las citas deben coincidir con el CSV y los resultados numéricos deben respetar sus filtros.

El gestor `08_gestor_skills.py` lee los archivos `SKILL.md`, decide cuáles aplicar y ejecuta una validación independiente de los resultados. Cuando la pregunta pide ejemplos y estadísticas, se activan ambas. Los archivos no representan dos nuevos servidores ni dos nuevas tools MCP.

In [8]:
import importlib

gestor_skills = importlib.import_module('08_gestor_skills')
skills = gestor_skills.cargar_skills()
print('Skills encontradas:')
for nombre, skill in skills.items():
    print(' -', nombre, '→', skill.descripcion)
print('Consulta combinada:', gestor_skills.seleccionar_skills(
    necesita_fuentes=True, necesita_estadisticas=True
))
subprocess.run([sys.executable, str(RAIZ / '09_probar_skills.py')], cwd=RAIZ, check=True)

Skills encontradas:
 - analisis-voz-consumidor → Interpreta opiniones reales de compradores y explica hallazgos cualitativos con respaldo documental.
 - auditoria-evidencia-rag → Comprueba trazabilidad de fuentes, integridad de resultados y precisión al utilizar estadísticas y citas de reseñas.
Consulta combinada: ['analisis-voz-consumidor', 'auditoria-evidencia-rag']


CompletedProcess(args=['C:\\Users\\bairo\\Amazon_MCP_RAG\\.venv\\Scripts\\python.exe', 'C:\\Users\\bairo\\Amazon_MCP_RAG\\09_probar_skills.py'], returncode=0)

## 7. El asistente conversacional

Gradio presenta el chat, Qwen3 elige las herramientas apropiadas y el programa aplica controles de calidad antes de responder. En particular, una solicitud de una reseña de cuatro estrellas no debe convertirse en una estadística. Las preguntas de seguimiento sencillas, como contar las vocales de una reseña citada, aprovechan el historial de la conversación.

La siguiente celda abre **la interfaz que se utiliza en el video**. Si el puerto `7860` ya está ocupado por otra instancia de Gradio, debe cerrarse esa instancia con `chat.close()` desde el notebook que la inició; no es necesario cerrar MCP.

In [9]:
import runpy
from IPython.display import display, Markdown

if puerto_activo('127.0.0.1', 7860):
    raise RuntimeError('Gradio ya está abierto en el puerto 7860; cierra su instancia anterior con chat.close().')

modulo_gradio = runpy.run_path(str(RAIZ / '07_app_gradio_v4_skills.py'), run_name='amazon_gradio_notebook')
chat = modulo_gradio['chat']
servidor_gradio = chat.launch(
    server_name='127.0.0.1', server_port=7860, share=False,
    prevent_thread_lock=True, inbrowser=False, show_error=True
)
display(Markdown('**Asistente disponible:** [abrir http://127.0.0.1:7860](http://127.0.0.1:7860)'))

C:\Users\bairo\Amazon_MCP_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\bairo\Amazon_MCP_RAG\.venv\Lib\site-packages\gradio\chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'messages', will be used.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


**Asistente disponible:** [abrir http://127.0.0.1:7860](http://127.0.0.1:7860)

[SKILLS] Seleccionadas: analisis-voz-consumidor, auditoria-evidencia-rag
[CONTROL] top_k de búsqueda corregido
[MCP | elección Qwen] buscar_resenas: {"pregunta": "Busca una reseña de 4 estrellas y explícame qué significa", "top_k": 1, "estrellas": [4]}
[AUDITORIA] 1 fuentes contrastadas con sus IDs originales
[SKILLS] Seleccionadas: auditoria-evidencia-rag
[MCP | elección Qwen] estadisticas_resenas: {"estrellas": [5], "palabra_clave": "calidad"}
[AUDITORIA] Conteo, filtros literales y porcentajes comprobados
[SKILLS] Seleccionadas: analisis-voz-consumidor, auditoria-evidencia-rag
[MCP | elección Qwen] estadisticas_resenas: {"estrellas": [1], "palabra_clave": "entrega"}
[CONTROL] top_k de búsqueda corregido
[MCP | elección Qwen] buscar_resenas: {"pregunta": "muéstrame dos quejas reales sobre retrasos y explica qué molestias expresan los consumidores.", "top_k": 2, "estrellas": null}
[AUDITORIA] Conteo, filtros literales y porcentajes comprobados | 2 fuentes contrastadas con sus IDs orig

## 8. Decisiones y ajustes durante el desarrollo

Las primeras versiones funcionaban para recuperar documentos, pero algunas preguntas revelaron dificultades de interpretación. Los cambios se introdujeron de forma gradual, conservando el servidor MCP con dos herramientas.

| Situación observada | Ajuste que se incorporó |
|---|---|
| Un saludo activaba FAISS sin necesidad. | Respuesta social directa para saludos y mensajes breves. |
| Una reseña solicitada por calificación disparaba estadísticas. | Separación entre filtro por estrellas y petición de conteo. |
| Qwen enviaba `estrellas: 4` en vez de `estrellas: [4]`. | Validación y corrección de parámetros antes de llamar la tool. |
| Se perdía la referencia a «esa misma reseña». | Lectura del historial y resolución del ID citado anteriormente. |
| Una síntesis llegó a contradecir un conteo confirmado por MCP. | Separación de la petición cualitativa, control de contradicciones y respaldo basado en las fuentes. |
| Algunos textos no fueron suficientemente específicos. | Skill de voz del consumidor y comprobación complementaria sobre textos originales. |

Los controles automáticos verifican IDs, textos, calificaciones, operaciones aritméticas y filtros literales. **La relevancia semántica y la calidad de las paráfrasis aún requieren revisión humana.**

## 9. Resultados y límites de la propuesta

La solución integra en un mismo asistente tres necesidades que antes requerían operaciones separadas: localizar testimonios, obtener estadísticas exactas e interpretar experiencias individuales. También puede conservar referencias simples de turnos anteriores.

La aplicación corre en un equipo de escritorio con memoria y GPU limitadas, sin depender de un servicio de inferencia en la nube. El entorno se ejecutó con una NVIDIA GTX 1660 SUPER de 6 GB de VRAM y 8 GB de RAM; la velocidad depende de las tareas y del estado del equipo.

Hay límites importantes: el conjunto contiene solamente 1.000 reseñas balanceadas; la búsqueda semántica devuelve documentos relevantes, pero no garantiza cubrir todas las opiniones; las estadísticas literales y los documentos semánticos no son necesariamente el mismo subconjunto; el historial permite algunas referencias sencillas, no memoria conversacional general; y la generación puede producir explicaciones deficientes que deben verificarse. No se consultan datos personales que no existan en el corpus.